In [1]:
# IMPORTS
import os
import uuid
from dotenv import load_dotenv

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

from sentence_transformers import CrossEncoder
from langchain_groq import ChatGroq

load_dotenv()

True

In [2]:
# Embeddings

model_name = "BAAI/bge-base-en-v1.5"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device":"cpu"},
    encode_kwargs={
        "normalize_embeddings":True
    }
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# chunking function

from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(docs):

    # Step 1 — structural split (Article level)
    structure_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200,
        separators=[
            "\nArticle ",
            "\nARTICLE ",
            "\nSection ",
            "\n\n"
        ]
    )

    structured_docs = structure_splitter.split_documents(docs)

    # Step 2 — semantic split
    semantic_splitter = SemanticChunker(
        embeddings
    )

    semantic_docs = semantic_splitter.split_documents(structured_docs)

    # Step 3 — size control
    final_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    final_docs = final_splitter.split_documents(semantic_docs)

    return final_docs

In [4]:
# Load Constitution Vector DB

from langchain_chroma import Chroma

DB_PATH = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database"

constitution_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="legal_knowledge"
)

print("Total constitution chunks:", constitution_db._collection.count())

Total constitution chunks: 787


In [7]:
# Temporary Upload DB

temp_doc_db = None

def load_uploaded_document(file_path):

    global temp_doc_db

    loader = PyPDFLoader(file_path)
    docs = loader.load()

    chunks = split_documents(docs)

    temp_doc_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    print("Uploaded document indexed:", len(chunks))

In [8]:
file_path = r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Mrs_Vandana_Dhirani_vs_Mrs_Arti_Kirloskar_on_27_July_2023.PDF'
load_uploaded_document(file_path)

Uploaded document indexed: 138


In [9]:
def retrieve_documents(query, k=10):

    results = []

    # Constitution retrieval
    const_docs = constitution_db.similarity_search(
        f"query: {query}",
        k=k
    )

    results.extend(const_docs)

    # Uploaded document retrieval
    if temp_doc_db is not None:

        doc_results = temp_doc_db.similarity_search(
            f"query: {query}",
            k=k
        )

        results.extend(doc_results)

    return results

In [10]:
reranker = CrossEncoder(
    "BAAI/bge-reranker-large"
)

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--BAAI--bge-reranker-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [11]:
def rerank(query, docs, top_k=5):

    pairs = [(query, d.page_content) for d in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc,_ in ranked[:top_k]]

In [12]:
def expand_query(query):

    return [
        query,
        f"legal interpretation of {query}",
        f"constitutional provisions related to {query}",
        f"relevant indian law regarding {query}"
    ]

In [13]:
def retrieve_pipeline(query):

    queries = expand_query(query)

    docs = []

    for q in queries:
        docs.extend(retrieve_documents(q))

    docs = rerank(query, docs)

    return docs

In [14]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [15]:
def legal_rag(query):

    docs = retrieve_pipeline(query)

    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""
You are an expert Indian legal assistant.

Answer the question using ONLY the provided legal context.

If the answer refers to an uploaded document, explain using that document.

If relevant constitutional provisions exist, cite the Article numbers.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [18]:
print(legal_rag(
    "I was caught jumping a red light and the police officer is asking for a 5,000 rupee fine. Is this amount correct?"
))

According to the provided context, there is no mention of traffic rules or fines for jumping a red light. The context only refers to constitutional provisions, such as Article 21 (Protection of life and personal liberty) and Article 22 (Protection against arrest and detention in certain cases), as well as the right to education (Article 21A). 

However, since the question involves a potential punishment, we can refer to the provision that "No person shall be prosecuted and punished for the same offence more than once" (Article not explicitly mentioned in the context, but it seems to be related to the Double Jeopardy clause). 

But, to determine the correctness of the fine amount, we would need to refer to a specific law or regulation related to traffic offenses, which is not provided in the given context. Therefore, based on the provided context, it is not possible to determine if the 5,000 rupee fine is correct.


In [19]:
load_uploaded_document(file_path)

legal_rag(
    "Does this contract violate Indian constitutional law?"
)

Uploaded document indexed: 138


"Based on the provided context, it appears that the contract in question is a lease agreement. The Indian constitutional law relevant to this case is Article 19, which deals with the protection of certain rights regarding freedom of speech, etc., and Article 14, which deals with equality before the law.\n\nThe contract's termination due to non-payment of rent by the defendant does not seem to directly violate Article 14 or Article 19 of the Indian Constitution. However, if the lease agreement was terminated or modified in a manner that is inconsistent with or abridges the rights conferred by Article 14 or Article 19, it may be deemed void under Article 31(e) of the Constitution.\n\nIn this case, since the non-payment of rent was the sole decision of the defendant, it does not seem to be a violation of the defendant's rights under Article 14 or Article 19. The court's decision to not equate the non-payment of rent with the happening of an event under Section 111(b) also suggests that th